In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install -q pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 13.0 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [3]:
!pip install -q rwkv==0.8.31

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.9/410.9 kB 11.4 MB/s eta 0:00:0000:01


In [31]:
!pip install -qi https://test.pypi.org/simple/ rwkvx==0.2.0.dev5

In [5]:
!pip install -q rwkv huggingface_hub

In [6]:
import os

import huggingface_hub

In [7]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

In [33]:
from rwkvx.generation import (
    RWKVSession,
    RWKVTokenizer,
    RWKVModel,
    RWKVSampler,
    GenerationConfig
)

from rwkvx.conversation import (
    ConversationManager,
    FormatPromptTemplate,
    HistoryManager,
    Message,
    OutputParser,
    PromptRenderer,
)

In [9]:
model_title = "rwkv7-g1b-2.9b-20251205-ctx8192"
# model_title = "rwkv7-g0a4-7.2b-20251208-ctx8192"
# model_title = "rwkv7-g0b-13.3b-20251130-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")

rwkv7-g1b-2.9b-20251205-ctx8192.pth:   0%|          | 0.00/5.90G [00:00<?, ?B/s]

In [10]:
config = GenerationConfig(
    max_new_tokens=500,
    stop_tokens=[0, 261],  # 0, \n\n
)

print(config)

GenerationConfig(max_new_tokens=500, temperature=1.0, top_p=0.3, top_k=0, presence_penalty=0.5, frequency_penalty=0.5, decay_penalty=0.996, stop_tokens=[0, 261])


In [11]:
model = RWKVModel(model_path=model_path)

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/0ca5368635c048d847f17be18454dbef318f2081/rwkv7-g1b-2.9b-20251205-ctx8192 (cuda fp16)



In [12]:
tokenizer = RWKVTokenizer(model)
sampler = RWKVSampler(config)

In [13]:
templates = {
    'system': FormatPromptTemplate('System: {text}\n\n'),
    'user': FormatPromptTemplate('User: {text}{think_mode}\n\n'),
    'assistant': FormatPromptTemplate('Assistant: {think_start}{think_text}{think_end}{text}\n\n'),
}

history = HistoryManager()

mgr = ConversationManager(
    templates=templates,
    history=history,
)

In [34]:
session = RWKVSession(model, tokenizer, sampler, mgr, config)

In [35]:
session.reset()
mgr.build_rwkv7_prompt()

''

In [36]:
mgr.add_system('Include emojis to all your answers.')

In [37]:
mgr.add_user('Tell me a short joke.')
mgr.build_rwkv7_prompt(
    auto_dummy=True
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think'

In [38]:
joke2 = session.generate_message()
print(joke2)

role='assistant' text='\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [39]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\n'

In [40]:
mgr.add_user('Tell me a joke different from previous one.')
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think'

In [41]:
joke = session.generate_message()
print(joke)

role='assistant' text='\n😄 Here’s a different one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [42]:
mgr.add_user('Tell me more complex joke.')
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s a different one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think'

In [43]:
joke3 = session.generate_message()
print(joke3)

role='assistant' text='\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions! 📚😭' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [44]:
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s a different one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions! 📚😭\n\n'

In [45]:
mgr.add_user('List topics of all jokes that you have told me. Explain the second joke.', metadata={'think_mode': ' think'})
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s a different one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions! 📚😭\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think'

In [46]:
final_answer = session.generate_message()

In [47]:
print(final_answer.metadata['think_text'])

We need to list all jokes told so far. The user has asked for "List topics of all jokes that you have told me." We have told three jokes:
1. A short joke about a tomato turning red because it saw salad dressing. Topic: food, humor, pun.
2. A joke about why the scarecrow won an award: outstanding in his field. Topic: agriculture, pun on "outstanding".
3. A more complex joke about math book being sad because of problems and no solutions. Topic: mathematics, humor, pun on "problems" and "solutions".
We need to explain the second joke (the scarecrow). Explanation should be clear and concise. Provide explanation of the pun and why it's funny. Also maybe mention that it's a play on words with "outstanding" meaning both impressive and standing out in a field (agriculture). So we can explain that the scarecrow is awarded because it stands out in its field (farm), hence "outstanding". It's a simple pun on "outstanding" meaning both impressive and standing out in a field (agriculture). That's th

In [48]:
print(final_answer.text)

### Jokes I’ve Shared So Far  
| # | Joke | Topic / Pun |
|---|------|-------------|
| 1 | **“Why did the tomato turn red?”** | Food‑related wordplay – “saw” = “saw” + “salad dressing.” |
| 2 | **“Why did the scarecrow win an award?”** | Agriculture‑themed pun – “outstanding” = both “impressive” and “standing out in a field.” |
| 3 | **“Why did the math book look sad?”** | Mathematics‑themed wordplay – “problems” + “solutions.” |
---
### Explanation of the Second Joke  
The second joke is a classic **pun**:  
> *“Why did the scarecrow win an award? Because he was outstanding in his field!”*  
- **Outstanding** can mean two things:  
  1. **Impressive or remarkable** – as in, the scarecrow looks good enough to be noticed.  
  2. **Standing out in a particular place or area** – here, it’s literally standing out *in* a field (the farm).  
By combining


In [49]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Here’s a different one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions! 📚😭\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>We need to list all jokes told so far. The user has asked for "List topics of all jokes that you have told me." We have told three jokes:\n1. A short joke about a tomato turning red because it saw salad dressing. Topic: food, humor, pun.\n2. A joke about why the 

In [50]:
# inspect think
mgr.history.get_message_by_number(8)

AttributeError: 'HistoryManager' object has no attribute 'get_message_by_number'

In [ ]:
mgr.add_user('Lets wrap up. Thank you!')
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

In [ ]:
endd = session.generate_message()
print(endd)